In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


def generate_apple_sales_data_with_promo_adjustment(
    base_demand: int = 1000, n_rows: int = 5000
):
    """
    Generates a synthetic dataset for predicting apple sales demand with seasonality
    and inflation.

    This function creates a pandas DataFrame with features relevant to apple sales.
    The features include date, average_temperature, rainfall, weekend flag, holiday flag,
    promotional flag, price_per_kg, and the previous day's demand. The target variable,
    'demand', is generated based on a combination of these features with some added noise.

    Args:
        base_demand (int, optional): Base demand for apples. Defaults to 1000.
        n_rows (int, optional): Number of rows (days) of data to generate. Defaults to 5000.

    Returns:
        pd.DataFrame: DataFrame with features and target variable for apple sales prediction.

    Example:
        >>> df = generate_apple_sales_data_with_seasonality(base_demand=1200, n_rows=6000)
        >>> df.head()
    """

    # Set seed for reproducibility
    np.random.seed(9999)

    # Create date range
    dates = [datetime.now() - timedelta(days=i) for i in range(n_rows)]
    dates.reverse()

    # Generate features
    df = pd.DataFrame(
        {
            "date": dates,
            "average_temperature": np.random.uniform(10, 35, n_rows),
            "rainfall": np.random.exponential(5, n_rows),
            "weekend": [(date.weekday() >= 5) * 1 for date in dates],
            "holiday": np.random.choice([0, 1], n_rows, p=[0.97, 0.03]),
            "price_per_kg": np.random.uniform(0.5, 3, n_rows),
            "month": [date.month for date in dates],
        }
    )

    # Introduce inflation over time (years)
    df["inflation_multiplier"] = (
        1 + (df["date"].dt.year - df["date"].dt.year.min()) * 0.03
    )

    # Incorporate seasonality due to apple harvests
    df["harvest_effect"] = np.sin(2 * np.pi * (df["month"] - 3) / 12) + np.sin(
        2 * np.pi * (df["month"] - 9) / 12
    )

    # Modify the price_per_kg based on harvest effect
    df["price_per_kg"] = df["price_per_kg"] - df["harvest_effect"] * 0.5

    # Adjust promo periods to coincide with periods lagging peak harvest by 1 month
    peak_months = [4, 10]  # months following the peak availability
    df["promo"] = np.where(
        df["month"].isin(peak_months),
        1,
        np.random.choice([0, 1], n_rows, p=[0.85, 0.15]),
    )

    # Generate target variable based on features
    base_price_effect = -df["price_per_kg"] * 50
    seasonality_effect = df["harvest_effect"] * 50
    promo_effect = df["promo"] * 200

    df["demand"] = (
        base_demand
        + base_price_effect
        + seasonality_effect
        + promo_effect
        + df["weekend"] * 300
        + np.random.normal(0, 50, n_rows)
    ) * df[
        "inflation_multiplier"
    ]  # adding random noise

    # Add previous day's demand
    df["previous_days_demand"] = df["demand"].shift(1)
    df["previous_days_demand"].fillna(
        method="bfill", inplace=True
    )  # fill the first row

    # Drop temporary columns
    df.drop(columns=["inflation_multiplier", "harvest_effect", "month"], inplace=True)

    return df

In [11]:
data = generate_apple_sales_data_with_promo_adjustment(base_demand=1_000, n_rows=1_000)

data[-20:]

C:\Users\Anuar\AppData\Local\Temp\ipykernel_52276\3859398416.py:89: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["previous_days_demand"].fillna(
C:\Users\Anuar\AppData\Local\Temp\ipykernel_52276\3859398416.py:89: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["previous_days_demand"].fillna(


,date,average_temperature,rainfall,weekend,holiday,price_per_kg,promo,demand,previous_days_demand
980,2025-10-13 10:53:54.581103,34.130183,1.454065,0,0,1.449177,1,1183.802447,1531.085782
981,2025-10-14 10:53:54.581103,32.353643,9.462859,0,0,2.856503,1,1030.951553,1183.802447
982,2025-10-15 10:53:54.581103,18.816833,0.391470,0,0,1.326429,1,1175.352029,1030.951553
983,2025-10-16 10:53:54.581103,34.533012,2.120477,0,0,0.970131,1,1251.385504,1175.352029
984,2025-10-17 10:53:54.581103,23.057202,2.365705,0,0,1.049931,1,1203.427049,1251.385504
985,2025-10-18 10:53:54.581103,34.810165,3.089005,1,0,2.035149,1,1504.971149,1203.427049
986,2025-10-19 10:53:54.581103,29.208905,3.673292,1,0,2.518098,1,1586.249547,1504.971149
987,2025-10-20 10:53:54.581103,16.428676,4.077782,0,0,1.268979,1,1275.118915,1586.249547
988,2025-10-21 10:53:54.581103,32.067512,2.734454,0,0,0.762317,1,1252.492007,1275.118915
989,2025-10-22 10:53:54.581103,31.938203,13.883486,0,0,1.153301,1,1179.040470,1252.492007


In [6]:
import os

In [ ]:
#Guardar DATASET usando ruta completa
# Corrección usando cadena cruda (r"")
ruta_external = r"C:\Users\Anuar\Documents\Maestria Inteligencia artificial\MLOPs Prueba\Cookie cutter\proyecto_mlops_equipo_56_fase_2\data\external"

# Guarda como CSV (Asegúrate de agregar un nombre de archivo al final de la ruta)
data.to_csv(ruta_external + r"\dataset_apples.csv", index=False)

In [29]:
from pathlib import Path

# 1. Definir la ruta relativa a la carpeta de destino
# Path("data") / "external" crea una ruta multiplataforma
# ("..")  Esto indica donde buscar la carpeta, en este caso un lugar arriba
ruta_carpeta_destino = Path("..") / "data" / "external"

# 2. Definir la ruta COMPLETA del archivo
ruta_final_archivo = ruta_carpeta_destino / "dataset_apples2.csv"

print(ruta_final_archivo)

# 3. CREAR EL DIRECTORIO si no existe
# .mkdir(parents=True, exist_ok=True) es la clave:
# - parents=True: Crea los directorios intermedios necesarios (ej: si no existe 'data', lo crea).
# - exist_ok=True: No lanza error si la carpeta ya existe.

print(f"Directorio creado o verificado: {ruta_carpeta_destino.resolve()}")

# 4. Guardar el dataset
# data.to_csv ahora usa el objeto Path, que pandas maneja perfectamente.
data.to_csv(ruta_final_archivo, index=False)

print(f"Dataset guardado exitosamente en: {ruta_final_archivo}")

..\data\external\dataset_apples2.csv
Directorio creado o verificado: C:\Users\Anuar\Documents\Maestria Inteligencia artificial\MLOPs Prueba\Cookie cutter\proyecto_mlops_equipo_56_fase_2\data\external
Dataset guardado exitosamente en: ..\data\external\dataset_apples2.csv


In [27]:
import os
print(f"Directorio de Trabajo Actual (CWD): {os.getcwd()}")

Directorio de Trabajo Actual (CWD): c:\Users\Anuar\Documents\Maestria Inteligencia artificial\MLOPs Prueba\Cookie cutter\proyecto_mlops_equipo_56_fase_2\mlops_project
